# Earthquake Seismic Analysis Pegasus Workflow

Earthquake monitoring and seismic hazard assessment are critical for public safety and scientific research. The [USGS Earthquake Hazards Program](https://earthquake.usgs.gov/) provides real-time and historical earthquake data through the FDSNWS Event API, covering global seismicity with detailed event parameters.

This workflow fetches earthquake catalog data from the USGS API, performs comprehensive seismic analysis including pattern recognition, anomaly detection, spatial clustering, aftershock prediction (statistical + ML), probabilistic seismic hazard assessment (PSHA), and seismic gap analysis.

The workflow processes each region through an 11-step sequential pipeline:

## Containers

All tools required to execute the jobs are included in a single Apptainer container built from the definition file in this repository:

`Apptainer/Earthquake_Container.def`, built to a `.sif` that Pegasus stages to the worker nodes (no registry pull), with:
* pandas, numpy, matplotlib, scipy, requests (base analysis)
* scikit-learn (clustering and ML)
* pytz (timezone handling)

## Accessing the Input Data

Earthquake data is fetched from the **USGS FDSNWS Event API** (`https://earthquake.usgs.gov/fdsnws/event/1/`). No API key is required. Rate limit is approximately 600 requests per minute with a maximum of 20,000 events per query.

**Predefined regions:** `pacific_ring`, `california`, `japan`, `indonesia`, `turkey`, `chile`, `worldwide`

## Workflow

The workflow processes earthquake data through an 11-step pipeline per region:

| Job Label                          | Description                                                                          |
|------------------------------------|--------------------------------------------------------------------------------------|
| fetch_earthquake_data              | Fetches earthquake catalog from USGS API for a region and date range                 |
| analyze_seismic_patterns           | Gutenberg-Richter b-value, depth profile, temporal trends, spatial distribution      |
| visualize_earthquakes              | Geographic map, magnitude-depth scatter, time series, cumulative magnitude plots      |
| detect_seismic_anomalies           | Swarm detection, mainshock-aftershock sequences, rate changes, depth anomalies        |
| cluster_seismic_zones              | Spatial clustering via DBSCAN, K-Means, or Hierarchical methods                      |
| predict_aftershocks                | Omori-Utsu decay, Bath's Law, Gutenberg-Richter; ML (Random Forest) predictions      |
| visualize_aftershock_predictions   | Mainshock maps with aftershock zones, probability charts, Omori decay curves          |
| assess_seismic_hazard              | Probabilistic Seismic Hazard Analysis (PSHA) with NGA-West2 GMPEs                    |
| analyze_seismic_gaps               | Temporal quiescence detection, moment deficit estimation, potential magnitude calc    |
| visualize_seismic_hazard           | PGA hazard maps, risk distribution, hazard curves for return periods                  |
| visualize_seismic_gaps             | Gap location maps, rate ratio heatmaps, risk scores, potential magnitude charts       |

## 1. Create the Earthquake Analysis Workflow

First, configure the region(s), date range, magnitude threshold, and analysis parameters.

Then the workflow class below will:
1. Build the Pegasus catalogs (sites, transformations, replicas)
2. Construct the DAG with the 11-step pipeline for each region
3. Data is fetched from USGS at runtime by the `fetch_earthquake_data` job

In [ ]:
# Regions to analyze (choices: pacific_ring, california, japan, indonesia, turkey, chile, worldwide)
REGIONS = ['california']

# Date range for earthquake data
START_DATE = '1994-01-01'
END_DATE = '1994-01-31'

# Minimum magnitude threshold
MIN_MAGNITUDE = 3.0

# Clustering parameters
CLUSTER_METHOD = 'dbscan'       # choices: dbscan, kmeans, hierarchical
CLUSTER_EPS = 50.0              # DBSCAN: max distance (km) between samples
CLUSTER_MIN_SAMPLES = 10        # DBSCAN: min samples for core points
CLUSTER_N_CLUSTERS = 5          # K-Means/Hierarchical: number of clusters

# Aftershock prediction parameters
AFTERSHOCK_THRESHOLD = 5.0      # Minimum magnitude for mainshock identification
AFTERSHOCK_TIME_WINDOWS = [1, 7, 30]  # Prediction windows in days

# Seismic hazard assessment parameters
HAZARD_GRID_RESOLUTION = 1.0    # Grid resolution in degrees
HAZARD_PGA_THRESHOLDS = [0.1, 0.2, 0.4]  # PGA thresholds in g

# Seismic gap analysis parameters
GAP_HISTORICAL_YEARS = 20       # Historical period for comparison
GAP_RECENT_YEARS = 5            # Recent period for comparison
GAP_RATE_THRESHOLD = 0.3        # Rate ratio threshold for gap detection

**Note:** For the 1994 Northridge earthquake analysis, use `california` region with dates `1994-01-01` to `1994-01-31` and `MIN_MAGNITUDE = 3.0`. For global significant earthquakes, use `worldwide` with `MIN_MAGNITUDE = 6.0`.

## Build the Apptainer container

The workflow runs every job inside `Apptainer/Earthquake_Container.sif`. Build it once here.
`Bootstrap: docker` in the definition file makes Apptainer pull and convert the
base image itself, so **no Docker installation and no registry login are needed**,
and Pegasus stages the resulting `.sif` to the worker nodes (`image_site="local"`).

A `.sif` carries a single architecture, so build it on this cluster rather than
copying one from a laptop. Skip this cell if the image already exists.


In [ ]:
# Build only if missing -- the build takes several minutes.
!test -f Apptainer/Earthquake_Container.sif || apptainer build Apptainer/Earthquake_Container.sif Apptainer/Earthquake_Container.def

# Confirm the image is usable and has what PegasusLite needs.
!apptainer exec Apptainer/Earthquake_Container.sif which curl wget
!apptainer exec Apptainer/Earthquake_Container.sif python -c "import pandas, sklearn, scipy; print('deps ok')"


In [ ]:
import os
import sys
import logging
from pathlib import Path
from datetime import datetime, timedelta

# --- Import Pegasus API ---
from Pegasus.api import *
logging.basicConfig(level=logging.DEBUG)


# --- Main workflow class ---
class EarthquakeWorkflow():
    wf = None
    sc = None
    tc = None
    rc = None
    props = None

    dagfile = None
    wf_dir = None
    shared_scratch_dir = None
    local_storage_dir = None
    wf_name = "earthquake"

    # --- Init ---
    def __init__(self, dagfile="workflow.yml"):
        self.dagfile = dagfile
        self.wf_dir = str(Path(".").resolve())
        self.shared_scratch_dir = os.path.join(self.wf_dir, "scratch")
        self.local_storage_dir = os.path.join(self.wf_dir, "output")

    # --- Write files in directory ---
    def write(self):
        if self.sc is not None:
            self.sc.write()
        self.props.write()
        self.rc.write()
        self.tc.write()

        try:
            self.wf.write(file=self.dagfile)
        except PegasusClientError as e:
            print(e)

    # --- Plan and Submit the workflow ---
    def plan_submit(self):
        try:
            self.wf.plan(submit=True)
        except PegasusClientError as e:
            print(e)

    # --- Get status of the workflow ---
    def status(self):
        try:
            self.wf.status(long=True)
        except PegasusClientError as e:
            print(e)

    # --- Wait for the workflow to finish ---
    def wait(self):
        try:
            self.wf.wait()
        except PegasusClientError as e:
            print(e)

    # --- Get statistics of the workflow ---
    def statistics(self):
        try:
            self.wf.statistics()
        except PegasusClientError as e:
            print(e)

    # --- Configuration (Pegasus Properties) ---
    def create_pegasus_properties(self):
        self.props = Properties()
        self.props["pegasus.transfer.threads"] = "16"
        return

    # --- Site Catalog ---
    def create_sites_catalog(self, exec_site_name="condorpool"):
        self.sc = SiteCatalog()

        local = (Site("local")
                    .add_directories(
                        Directory(Directory.SHARED_SCRATCH, self.shared_scratch_dir)
                            .add_file_servers(FileServer("file://" + self.shared_scratch_dir, Operation.ALL)),
                        Directory(Directory.LOCAL_STORAGE, self.local_storage_dir)
                            .add_file_servers(FileServer("file://" + self.local_storage_dir, Operation.ALL))
                    )
                )

        exec_site = (Site(exec_site_name)
                        .add_condor_profile(universe="vanilla")
                        .add_pegasus_profile(style="condor")
                    )

        self.sc.add_sites(local, exec_site)

    # --- Transformation Catalog (Executables and Containers) ---
    def create_transformation_catalog(self, exec_site_name="condorpool"):
        self.tc = TransformationCatalog()

        # A locally built Apptainer image. Pegasus stages the .sif like any other
        # input file, so image_site is "local" -- where the file physically lives.
        sif = os.path.join(self.wf_dir, "Apptainer/Earthquake_Container.sif")
        if not os.path.exists(sif):
            print(f"Warning: {sif} not found -- run the build cell above first")

        earthquake_container = Container("earthquake_container",
            container_type=Container.SINGULARITY,
            image="file://" + sif,
            image_site="local"
        )

        # Transformations
        fetch_earthquake_data = Transformation("fetch_earthquake_data", site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/fetch_earthquake_data.py"),
            is_stageable=True, container=earthquake_container
        ).add_pegasus_profile(memory="2 GB")

        analyze_seismic_patterns = Transformation("analyze_seismic_patterns", site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/analyze_seismic_patterns.py"),
            is_stageable=True, container=earthquake_container
        ).add_pegasus_profile(memory="2 GB")

        visualize_earthquakes = Transformation("visualize_earthquakes", site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/visualize_earthquakes.py"),
            is_stageable=True, container=earthquake_container
        ).add_pegasus_profile(memory="2 GB")

        detect_seismic_anomalies = Transformation("detect_seismic_anomalies", site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/detect_seismic_anomalies.py"),
            is_stageable=True, container=earthquake_container
        ).add_pegasus_profile(memory="2 GB")

        cluster_seismic_zones = Transformation("cluster_seismic_zones", site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/cluster_seismic_zones.py"),
            is_stageable=True, container=earthquake_container
        ).add_pegasus_profile(memory="2 GB")

        predict_aftershocks = Transformation("predict_aftershocks", site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/predict_aftershocks.py"),
            is_stageable=True, container=earthquake_container
        ).add_pegasus_profile(memory="4 GB")

        visualize_aftershock_predictions = Transformation("visualize_aftershock_predictions", site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/visualize_aftershock_predictions.py"),
            is_stageable=True, container=earthquake_container
        ).add_pegasus_profile(memory="2 GB")

        assess_seismic_hazard = Transformation("assess_seismic_hazard", site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/assess_seismic_hazard.py"),
            is_stageable=True, container=earthquake_container
        ).add_pegasus_profile(memory="2 GB")

        analyze_seismic_gaps = Transformation("analyze_seismic_gaps", site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/analyze_seismic_gaps.py"),
            is_stageable=True, container=earthquake_container
        ).add_pegasus_profile(memory="2 GB")

        visualize_seismic_hazard = Transformation("visualize_seismic_hazard", site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/visualize_seismic_hazard.py"),
            is_stageable=True, container=earthquake_container
        ).add_pegasus_profile(memory="2 GB")

        visualize_seismic_gaps = Transformation("visualize_seismic_gaps", site=exec_site_name,
            pfn=os.path.join(self.wf_dir, "bin/visualize_seismic_gaps.py"),
            is_stageable=True, container=earthquake_container
        ).add_pegasus_profile(memory="2 GB")

        self.tc.add_containers(earthquake_container)
        self.tc.add_transformations(
            fetch_earthquake_data, analyze_seismic_patterns, visualize_earthquakes,
            detect_seismic_anomalies, cluster_seismic_zones, predict_aftershocks,
            visualize_aftershock_predictions, assess_seismic_hazard, analyze_seismic_gaps,
            visualize_seismic_hazard, visualize_seismic_gaps
        )

    # --- Replica Catalog ---
    def create_replica_catalog(self):
        self.rc = ReplicaCatalog()
        # No input files needed - fetch_earthquake_data fetches from USGS API at runtime

    # --- Create Workflow ---
    def create_workflow(self, regions, start_date, end_date, min_magnitude,
                        cluster_method="dbscan", cluster_eps=50.0,
                        cluster_min_samples=10, cluster_n_clusters=5,
                        aftershock_threshold=5.0, aftershock_time_windows=[1, 7, 30],
                        hazard_grid_resolution=1.0, hazard_pga_thresholds=[0.1, 0.2, 0.4],
                        gap_historical_years=20, gap_recent_years=5, gap_rate_threshold=0.3):
        self.wf = Workflow(self.wf_name, infer_dependencies=True)

        for region in regions:
            self._add_region_jobs(region, start_date, end_date, min_magnitude,
                                 cluster_method, cluster_eps,
                                 cluster_min_samples, cluster_n_clusters,
                                 aftershock_threshold, aftershock_time_windows,
                                 hazard_grid_resolution, hazard_pga_thresholds,
                                 gap_historical_years, gap_recent_years, gap_rate_threshold)

    def _add_region_jobs(self, region, start_date, end_date, min_magnitude,
                        cluster_method, cluster_eps, cluster_min_samples,
                        cluster_n_clusters, aftershock_threshold, aftershock_time_windows,
                        hazard_grid_resolution, hazard_pga_thresholds,
                        gap_historical_years, gap_recent_years, gap_rate_threshold):
        print(f"  Adding jobs for region: {region}")

        # Output files
        catalog_file = File(f"{region}_catalog.csv")
        patterns_file = File(f"{region}_patterns.json")
        visualization_file = File(f"{region}_visualization.png")
        anomalies_file = File(f"{region}_anomalies.json")
        zones_file = File(f"{region}_zones.json")
        aftershock_file = File(f"{region}_aftershock_predictions.json")
        aftershock_viz_file = File(f"{region}_aftershock_visualization.png")
        hazard_file = File(f"{region}_seismic_hazard.json")
        gaps_file = File(f"{region}_seismic_gaps.json")
        hazard_viz_file = File(f"{region}_hazard_visualization.png")
        gaps_viz_file = File(f"{region}_gaps_visualization.png")

        # Job 1: Fetch earthquake data
        fetch_job = (
            Job("fetch_earthquake_data", _id=f"fetch_{region}", node_label=f"fetch_{region}")
            .add_args("--region", region, "--start-date", start_date,
                      "--end-date", end_date, "--min-magnitude", str(min_magnitude),
                      "--output", catalog_file)
            .add_outputs(catalog_file, stage_out=True, register_replica=False)
            .add_pegasus_profiles(label=region)
        )
        self.wf.add_jobs(fetch_job)

        # Job 2: Analyze seismic patterns
        analyze_job = (
            Job("analyze_seismic_patterns", _id=f"analyze_{region}", node_label=f"analyze_{region}")
            .add_args("--input", catalog_file, "--output", patterns_file)
            .add_inputs(catalog_file)
            .add_outputs(patterns_file, stage_out=True, register_replica=False)
            .add_pegasus_profiles(label=region)
        )
        self.wf.add_jobs(analyze_job)

        # Job 3: Visualize earthquakes
        title = f"{region.title()}_Earthquakes"
        visualize_job = (
            Job("visualize_earthquakes", _id=f"visualize_{region}", node_label=f"visualize_{region}")
            .add_args("--input", catalog_file, "--output", visualization_file, "--title", title)
            .add_inputs(catalog_file)
            .add_outputs(visualization_file, stage_out=True, register_replica=False)
            .add_pegasus_profiles(label=region)
        )
        self.wf.add_jobs(visualize_job)

        # Job 4: Detect seismic anomalies
        anomalies_job = (
            Job("detect_seismic_anomalies", _id=f"anomalies_{region}", node_label=f"anomalies_{region}")
            .add_args("--input", catalog_file, "--output", anomalies_file)
            .add_inputs(catalog_file)
            .add_outputs(anomalies_file, stage_out=True, register_replica=False)
            .add_pegasus_profiles(label=region)
        )
        self.wf.add_jobs(anomalies_job)

        # Job 5: Cluster seismic zones
        cluster_args = ["--input", catalog_file, "--output", zones_file,
                        "--method", cluster_method]
        if cluster_method == "dbscan":
            cluster_args.extend(["--eps", str(cluster_eps),
                                "--min-samples", str(cluster_min_samples)])
        elif cluster_method in ("kmeans", "hierarchical"):
            cluster_args.extend(["--n-clusters", str(cluster_n_clusters)])

        cluster_job = (
            Job("cluster_seismic_zones", _id=f"cluster_{region}", node_label=f"cluster_{region}")
            .add_args(*cluster_args)
            .add_inputs(catalog_file)
            .add_outputs(zones_file, stage_out=True, register_replica=False)
            .add_pegasus_profiles(label=region)
        )
        self.wf.add_jobs(cluster_job)

        # Job 6: Predict aftershocks
        aftershock_args = ["--input", catalog_file, "--output", aftershock_file,
                           "--mainshock-threshold", str(aftershock_threshold),
                           "--time-windows"]
        aftershock_args.extend([str(w) for w in aftershock_time_windows])

        aftershock_job = (
            Job("predict_aftershocks", _id=f"aftershock_{region}", node_label=f"aftershock_{region}")
            .add_args(*aftershock_args)
            .add_inputs(catalog_file)
            .add_outputs(aftershock_file, stage_out=True, register_replica=False)
            .add_pegasus_profiles(label=region)
        )
        self.wf.add_jobs(aftershock_job)

        # Job 7: Visualize aftershock predictions
        aftershock_title = f"{region.title()}_Aftershock_Predictions"
        aftershock_viz_job = (
            Job("visualize_aftershock_predictions", _id=f"aftershock_viz_{region}", node_label=f"aftershock_viz_{region}")
            .add_args("--input", aftershock_file, "--catalog", catalog_file,
                      "--output", aftershock_viz_file, "--title", aftershock_title)
            .add_inputs(aftershock_file, catalog_file)
            .add_outputs(aftershock_viz_file, stage_out=True, register_replica=False)
            .add_pegasus_profiles(label=region)
        )
        self.wf.add_jobs(aftershock_viz_job)

        # Job 8: Assess seismic hazard
        hazard_args = ["--input", catalog_file, "--output", hazard_file,
                       "--grid-resolution", str(hazard_grid_resolution),
                       "--pga-thresholds"]
        hazard_args.extend([str(t) for t in hazard_pga_thresholds])

        hazard_job = (
            Job("assess_seismic_hazard", _id=f"hazard_{region}", node_label=f"hazard_{region}")
            .add_args(*hazard_args)
            .add_inputs(catalog_file)
            .add_outputs(hazard_file, stage_out=True, register_replica=False)
            .add_pegasus_profiles(label=region)
        )
        self.wf.add_jobs(hazard_job)

        # Job 9: Analyze seismic gaps
        gaps_job = (
            Job("analyze_seismic_gaps", _id=f"gaps_{region}", node_label=f"gaps_{region}")
            .add_args("--input", catalog_file, "--output", gaps_file,
                      "--historical-years", str(gap_historical_years),
                      "--recent-years", str(gap_recent_years),
                      "--rate-threshold", str(gap_rate_threshold))
            .add_inputs(catalog_file)
            .add_outputs(gaps_file, stage_out=True, register_replica=False)
            .add_pegasus_profiles(label=region)
        )
        self.wf.add_jobs(gaps_job)

        # Job 10: Visualize seismic hazard
        hazard_viz_title = f"{region.title()}_Seismic_Hazard"
        hazard_viz_job = (
            Job("visualize_seismic_hazard", _id=f"hazard_viz_{region}", node_label=f"hazard_viz_{region}")
            .add_args("--input", hazard_file, "--catalog", catalog_file,
                      "--output", hazard_viz_file, "--title", hazard_viz_title)
            .add_inputs(hazard_file, catalog_file)
            .add_outputs(hazard_viz_file, stage_out=True, register_replica=False)
            .add_pegasus_profiles(label=region)
        )
        self.wf.add_jobs(hazard_viz_job)

        # Job 11: Visualize seismic gaps
        gaps_viz_title = f"{region.title()}_Seismic_Gaps"
        gaps_viz_job = (
            Job("visualize_seismic_gaps", _id=f"gaps_viz_{region}", node_label=f"gaps_viz_{region}")
            .add_args("--input", gaps_file, "--catalog", catalog_file,
                      "--output", gaps_viz_file, "--title", gaps_viz_title)
            .add_inputs(gaps_file, catalog_file)
            .add_outputs(gaps_viz_file, stage_out=True, register_replica=False)
            .add_pegasus_profiles(label=region)
        )
        self.wf.add_jobs(gaps_viz_job)


# --- Build and generate the workflow ---
start_date = datetime.strptime(START_DATE, '%Y-%m-%d')
if END_DATE:
    end_date = datetime.strptime(END_DATE, '%Y-%m-%d')
else:
    end_date = start_date + timedelta(days=30)

dagfile = 'workflow.yml'

workflow = EarthquakeWorkflow(dagfile=dagfile)

print("Creating execution sites...")
workflow.create_sites_catalog("condorpool")

print("Creating workflow properties...")
workflow.create_pegasus_properties()

print("Creating transformation catalog...")
workflow.create_transformation_catalog("condorpool")

print("Creating replica catalog...")
workflow.create_replica_catalog()

print("Creating earthquake workflow DAG...")
workflow.create_workflow(
    regions=REGIONS,
    start_date=START_DATE,
    end_date=end_date.strftime('%Y-%m-%d'),
    min_magnitude=MIN_MAGNITUDE,
    cluster_method=CLUSTER_METHOD,
    cluster_eps=CLUSTER_EPS,
    cluster_min_samples=CLUSTER_MIN_SAMPLES,
    cluster_n_clusters=CLUSTER_N_CLUSTERS,
    aftershock_threshold=AFTERSHOCK_THRESHOLD,
    aftershock_time_windows=AFTERSHOCK_TIME_WINDOWS,
    hazard_grid_resolution=HAZARD_GRID_RESOLUTION,
    hazard_pga_thresholds=HAZARD_PGA_THRESHOLDS,
    gap_historical_years=GAP_HISTORICAL_YEARS,
    gap_recent_years=GAP_RECENT_YEARS,
    gap_rate_threshold=GAP_RATE_THRESHOLD
)

workflow.write()
print("\nEarthquake Analysis Workflow has been generated!")

## View the Generated Workflow DAG

Before submitting, we can visualize the workflow DAG using `pegasus-graphviz`. The graph shows all 11 jobs as nodes and data dependencies as edges. All analysis jobs depend on the initial `fetch_earthquake_data` job which retrieves the catalog from the USGS API.

In [ ]:
!pegasus-graphviz -f workflow.yml --output workflow.png

In [ ]:
from IPython.display import Image
Image(filename='workflow.png')

## 2. Plan and Submit the Workflow

We will now plan and submit the workflow for execution. By default we are running jobs on site **condorpool** i.e. the selected ACCESS resource.

In [ ]:
workflow.plan_submit()

After the workflow has been successfully planned and submitted, you can use the Python `Workflow` object to monitor the status of the workflow. It shows in detail the counts of jobs of each status and whether a job is idle or running.

In [ ]:
workflow.status()

In [ ]:
workflow.wait()

## 3. Statistics

Depending on whether the workflow finished successfully or not, you have options on what to do next. If the workflow failed you can use `workflow.analyze()` to get help finding out what went wrong. If the workflow finished successfully, we can pull out some statistics from the provenance database:

In [ ]:
workflow.statistics()

## 4. Examining the Results

Once the workflow has finished, we can look at the output directory for our results. The workflow produces the following outputs for each region:

```
output/
├── <region>_catalog.csv                      # Raw earthquake catalog from USGS
├── <region>_patterns.json                    # Gutenberg-Richter b-value, depth/temporal/spatial stats
├── <region>_visualization.png                # Multi-panel earthquake visualization
├── <region>_anomalies.json                   # Swarms, aftershock sequences, rate changes
├── <region>_zones.json                       # Spatial clustering results
├── <region>_aftershock_predictions.json      # Omori-Utsu + ML aftershock probabilities
├── <region>_aftershock_visualization.png     # Mainshock maps, probability charts, decay curves
├── <region>_seismic_hazard.json              # PSHA grid with PGA exceedance probabilities
├── <region>_hazard_visualization.png         # PGA hazard maps and risk distribution
├── <region>_seismic_gaps.json                # Quiescence zones with moment deficit
└── <region>_gaps_visualization.png           # Gap maps, rate ratios, potential magnitudes
```

In [ ]:
!ls -ltR output/

### Earthquake Visualization

The comprehensive visualization includes a geographic map (color-coded by depth, sized by magnitude), magnitude histogram, depth histogram, time series of daily event counts with 7-day rolling average, magnitude-depth scatter plot, and cumulative magnitude (Gutenberg-Richter) curve.

In [ ]:
import glob
from IPython.display import Image, display

viz_pngs = sorted(glob.glob("output/*_visualization.png"))
for png in viz_pngs:
    print(f"\n{png}")
    display(Image(filename=png))

### Aftershock Predictions

The aftershock visualization shows mainshock locations with risk-colored aftershock zones (red = high, orange = moderate, green = low), probability comparison bar charts for M4+/M5+/M6+ over 7-day windows, and Omori-Utsu decay curves for each identified mainshock.

In [ ]:
aftershock_pngs = sorted(glob.glob("output/*_aftershock_visualization.png"))
for png in aftershock_pngs:
    print(f"\n{png}")
    display(Image(filename=png))

### Seismic Hazard Assessment

The hazard visualization displays PGA (Peak Ground Acceleration) hazard maps with a colorscale from green (very low) through yellow/orange to dark red (very high risk), risk level distribution, and hazard curves showing exceedance probabilities for 50, 100, and 475-year return periods.

In [ ]:
hazard_pngs = sorted(glob.glob("output/*_hazard_visualization.png"))
for png in hazard_pngs:
    print(f"\n{png}")
    display(Image(filename=png))

### Seismic Gap Analysis

The gap analysis visualization shows regions of anomalous seismic quiescence where historical activity has significantly decreased in recent years. Maps display gap locations with risk scores, rate ratio heatmaps (low ratio = gap), and charts of potential magnitude based on accumulated moment deficit.

In [ ]:
gaps_pngs = sorted(glob.glob("output/*_gaps_visualization.png"))
for png in gaps_pngs:
    print(f"\n{png}")
    display(Image(filename=png))

### Anomaly Detection Summary

The anomaly detection results identify earthquake swarms (temporal+spatial clustering), mainshock-aftershock sequences (Omori law parameters), magnitude anomalies (Z-score > 2.5), seismicity rate changes (3σ above baseline), and unusual depth patterns.

In [ ]:
import json

anomaly_files = sorted(glob.glob("output/*_anomalies.json"))
for anomaly_file in anomaly_files:
    with open(anomaly_file, 'r') as f:
        anomaly_data = json.load(f)

    region = anomaly_data.get('region', anomaly_file.split('/')[-1].replace('_anomalies.json', ''))
    summary = anomaly_data.get('summary', {})

    print(f"\n--- {region} ---")
    print(f"  Total events analyzed:       {summary.get('total_events', 'N/A')}")
    print(f"  Swarms detected:             {summary.get('swarm_count', summary.get('total_swarms', 'N/A'))}")
    print(f"  Mainshock-aftershock seqs:    {summary.get('sequence_count', summary.get('total_sequences', 'N/A'))}")
    print(f"  Magnitude anomalies:         {summary.get('magnitude_anomaly_count', summary.get('total_magnitude_anomalies', 'N/A'))}")
    print(f"  Rate change anomalies:       {summary.get('rate_change_count', summary.get('total_rate_changes', 'N/A'))}")
    print(f"  Depth anomalies:             {summary.get('depth_anomaly_count', summary.get('total_depth_anomalies', 'N/A'))}")

### Seismic Pattern Analysis Summary

The pattern analysis includes Gutenberg-Richter b-value estimation, depth categorization (shallow < 70 km, intermediate 70-300 km, deep > 300 km), magnitude-depth correlation, temporal trends, and spatial distribution metrics.

In [ ]:
pattern_files = sorted(glob.glob("output/*_patterns.json"))
for pattern_file in pattern_files:
    with open(pattern_file, 'r') as f:
        pattern_data = json.load(f)

    region = pattern_file.split('/')[-1].replace('_patterns.json', '')
    mag_stats = pattern_data.get('magnitude_distribution', pattern_data.get('magnitude_statistics', {}))
    depth_stats = pattern_data.get('depth_profile', pattern_data.get('depth_statistics', {}))
    temporal = pattern_data.get('temporal_patterns', pattern_data.get('temporal_statistics', {}))

    print(f"\n--- {region} ---")
    print(f"  Total events:    {pattern_data.get('total_events', 'N/A')}")
    print(f"  Magnitude range: {mag_stats.get('min', mag_stats.get('min_magnitude', 'N/A'))} - "
          f"{mag_stats.get('max', mag_stats.get('max_magnitude', 'N/A'))}")
    print(f"  Mean magnitude:  {mag_stats.get('mean', mag_stats.get('mean_magnitude', 'N/A'))}")
    if 'b_value' in mag_stats or 'b_value' in pattern_data:
        b_val = mag_stats.get('b_value', pattern_data.get('b_value', 'N/A'))
        print(f"  G-R b-value:     {b_val}")
    print(f"  Depth range:     {depth_stats.get('min', depth_stats.get('min_depth', 'N/A'))} - "
          f"{depth_stats.get('max', depth_stats.get('max_depth', 'N/A'))} km")
    if temporal:
        print(f"  Daily event rate: {temporal.get('mean_daily_rate', temporal.get('daily_event_rate', 'N/A'))}")

### Aftershock Prediction Summary

The aftershock predictions use Omori-Utsu decay law and Bath's Law to estimate probabilities of aftershocks at magnitude thresholds (M4+, M5+, M6+) over 1-day, 7-day, and 30-day windows.

In [ ]:
aftershock_files = sorted(glob.glob("output/*_aftershock_predictions.json"))
for af_file in aftershock_files:
    with open(af_file, 'r') as f:
        af_data = json.load(f)

    region = af_file.split('/')[-1].replace('_aftershock_predictions.json', '')
    predictions = af_data.get('predictions', af_data.get('mainshock_predictions', []))

    print(f"\n--- {region} ---")
    print(f"  Mainshocks identified: {len(predictions)}")
    for pred in predictions[:5]:  # Show top 5
        ms = pred.get('mainshock', pred)
        mag = ms.get('magnitude', 'N/A')
        loc = ms.get('location', ms.get('place', 'N/A'))
        risk = pred.get('risk_level', pred.get('overall_risk', 'N/A'))
        print(f"    M{mag} at {loc} - Risk: {risk}")